# 5b: Graph Connectivity vs Trading Performance

Analyses the relationship between **graph density** (number of edges in the
rolling Pearson correlation graph) and **strategy performance** (rolling Sharpe ratio).

**Key question:** Is the visual alignment between connectivity spikes and Sharpe
movements statistically significant, or coincidence?

**Models analysed:** Rolling GCN, 4c GAT Rolling (both use time-varying graphs)

**Steps:**
1. Verify correctness of connectivity and rolling Sharpe computations
2. Visual analysis with event annotations
3. Statistical verification (correlation, lagged cross-correlation, Granger causality)
4. Regime analysis (high vs low connectivity)
5. VIX overlay (if available)
6. GCN vs GAT comparison

## 1. Setup

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")
print(f"Results base: {RESULTS_BASE}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.signal import correlate
print("Imports ready")

## 2. Load Data

In [ ]:
# Load rolling GCN results
from gml.experiment_utils import load_experiment_results

# Adjust these to match your saved experiment names and configs
GCN_EXPERIMENT = "3_GCN_rolling_pearson"
GCN_CONFIG = "lb20_th0.4_equity"  # Adjust to match your config name
GCN_SEED = 40

GAT_EXPERIMENT = "4c_GATv2_rolling_pearson"
GAT_CONFIG = "lb20_th0.4_equity"  # Must match GCN threshold for fair comparison
GAT_SEED = 40

print(f"Loading GCN: {GCN_EXPERIMENT}/{GCN_CONFIG}/seed_{GCN_SEED}")
gcn_results = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)

print(f"Loading GAT: {GAT_EXPERIMENT}/{GAT_CONFIG}/seed_{GAT_SEED}")
gat_results = load_experiment_results(GAT_EXPERIMENT, GAT_SEED, config_name=GAT_CONFIG, base_dir=RESULTS_BASE)

print("
GCN data loaded:", list(gcn_results.keys()))
print("GAT data loaded:", list(gat_results.keys()))

In [ ]:
# Extract adjacency matrices and daily returns
gcn_adj = gcn_results["adjacency"]       # (num_windows, 88, 88)
gcn_returns = gcn_results["daily_returns"]  # pd.Series indexed by date
gcn_dates = gcn_results["test_dates"]       # array of dates

gat_adj = gat_results.get("adjacency")     # Same Pearson mask as GCN (if saved)
gat_attn = gat_results.get("attention_weights")  # (num_windows, heads, 88, 88)
gat_returns = gat_results["daily_returns"]
gat_dates = gat_results["test_dates"]

print(f"GCN adjacency shape: {gcn_adj.shape}")
print(f"GCN daily returns: {len(gcn_returns)} days")
print(f"GCN test dates: {len(gcn_dates)} windows")
if gat_attn is not None:
    print(f"GAT attention shape: {gat_attn.shape}")

## 3. Step 1: Verify Correctness of Computations

### 3a. Recompute edge count from raw adjacency matrices

For each test window, count edges where adjacency > 0 (excluding self-loops).
This must match what was plotted in the training notebooks.

In [ ]:
def compute_edge_counts(adjacencies):
    """Recompute edge count per window from raw adjacency matrices."""
    edge_counts = []
    for adj in adjacencies:
        # Exclude self-loops (diagonal)
        adj_no_diag = adj.copy()
        np.fill_diagonal(adj_no_diag, 0)
        # Count non-zero edges (undirected: divide by 2)
        n_edges = (adj_no_diag > 0).sum() / 2
        edge_counts.append(n_edges)
    return np.array(edge_counts)

gcn_edges = compute_edge_counts(gcn_adj)

print(f"Edge count statistics (GCN rolling):")
print(f"  Mean: {gcn_edges.mean():.1f}")
print(f"  Std:  {gcn_edges.std():.1f}")
print(f"  Min:  {gcn_edges.min():.0f}")
print(f"  Max:  {gcn_edges.max():.0f}")

# Sanity check: plot edge counts over time
plt.figure(figsize=(14, 4))
plt.plot(pd.to_datetime(gcn_dates), gcn_edges, linewidth=0.8)
plt.xlabel("Date"); plt.ylabel("Number of Edges")
plt.title("GCN Rolling: Edge Count Over Time (recomputed from raw adjacency)")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 3b. Recompute rolling Sharpe from raw daily returns

Compute rolling Sharpe ratio with different window sizes to check robustness.

In [ ]:
def compute_rolling_sharpe(daily_returns, window):
    """Compute annualized rolling Sharpe ratio."""
    rolling_mean = daily_returns.rolling(window).mean()
    rolling_std = daily_returns.rolling(window).std()
    return (rolling_mean / rolling_std) * np.sqrt(252)

# Compute for multiple windows
gcn_sharpe_20 = compute_rolling_sharpe(gcn_returns, 20)
gcn_sharpe_60 = compute_rolling_sharpe(gcn_returns, 60)

# Sanity check: plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(gcn_sharpe_20.index, gcn_sharpe_20.values, linewidth=0.8, label="Window=20")
axes[0].plot(gcn_sharpe_60.index, gcn_sharpe_60.values, linewidth=0.8, label="Window=60")
axes[0].axhline(y=0, color="black", ls="--", lw=0.5)
axes[0].set_ylabel("Rolling Sharpe"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_title("GCN Rolling: Rolling Sharpe (recomputed from raw daily returns)")

axes[1].plot(gcn_returns.index, gcn_returns.cumsum().values, linewidth=0.8)
axes[1].set_ylabel("Cumulative Returns"); axes[1].grid(True, alpha=0.3)
axes[1].set_title("GCN Rolling: Cumulative Returns")
plt.tight_layout(); plt.show()

### 3c. Verify date alignment

Edge counts are per-window (one per sliding window). Daily returns are per-day.
We need to align them: each window corresponds to a specific date (the last day of the window).

In [ ]:
# Convert window dates to datetime
gcn_window_dates = pd.to_datetime(gcn_dates)

# Each window's edge count corresponds to the window's last date
# Daily returns are indexed by date
# We need to align: for each window date, get the corresponding daily return

# Check overlap
print(f"Window dates range: {gcn_window_dates[0].date()} to {gcn_window_dates[-1].date()}")
print(f"Daily returns range: {gcn_returns.index[0].date()} to {gcn_returns.index[-1].date()}")

# Create aligned dataframe
gcn_connectivity = pd.Series(gcn_edges, index=gcn_window_dates, name="edge_count")

# Align with rolling Sharpe (which is indexed by daily return dates)
aligned = pd.DataFrame({
    "edge_count": gcn_connectivity,
    "rolling_sharpe_20": gcn_sharpe_20,
    "rolling_sharpe_60": gcn_sharpe_60,
}).dropna()

print(f"
Aligned dataframe: {len(aligned)} rows")
print(f"Date range: {aligned.index[0].date()} to {aligned.index[-1].date()}")
aligned.head()

## 4. Step 2: Visual Analysis

In [ ]:
def plot_dual_axis(dates, connectivity, rolling_sharpe, title, events=None):
    """Dual-axis plot: connectivity (left) + rolling Sharpe (right)."""
    fig, ax1 = plt.subplots(figsize=(16, 6))

    color1 = "#1f77b4"
    ax1.plot(dates, connectivity, color=color1, linewidth=0.8, alpha=0.8)
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Number of Edges", color=color1)
    ax1.tick_params(axis="y", labelcolor=color1)
    ax1.grid(True, alpha=0.2)

    ax2 = ax1.twinx()
    color2 = "#d62728"
    ax2.plot(dates, rolling_sharpe, color=color2, linewidth=0.8, alpha=0.8)
    ax2.set_ylabel("Rolling Sharpe Ratio", color=color2)
    ax2.tick_params(axis="y", labelcolor=color2)
    ax2.axhline(y=0, color="black", ls="--", lw=0.5)

    # Annotate key events
    if events:
        for date, label in events.items():
            ax1.axvline(x=pd.Timestamp(date), color="gray", ls="--", lw=1, alpha=0.5)
            ax1.text(pd.Timestamp(date), ax1.get_ylim()[1] * 0.95, label,
                     fontsize=8, ha="center", color="gray")

    plt.title(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

# Key market events during test period (2017-2023)
events = {
    "2018-02-05": "VIX spike",
    "2020-03-16": "COVID crash",
    "2022-03-16": "Rate hikes begin",
    "2022-06-13": "Bear market",
}

# Plot for rolling window = 20
plot_dual_axis(
    aligned.index, aligned["edge_count"], aligned["rolling_sharpe_20"],
    "GCN Rolling: Graph Connectivity vs Rolling Sharpe (window=20)",
    events=events,
)

# Plot for rolling window = 60
plot_dual_axis(
    aligned.index, aligned["edge_count"], aligned["rolling_sharpe_60"],
    "GCN Rolling: Graph Connectivity vs Rolling Sharpe (window=60)",
    events=events,
)

## 5. Step 3: Statistical Verification

### 5a. Pearson Correlation

In [ ]:
# Concurrent correlation
for window_name, sharpe_col in [("20-day", "rolling_sharpe_20"), ("60-day", "rolling_sharpe_60")]:
    corr, pval = stats.pearsonr(aligned["edge_count"], aligned[sharpe_col])
    print(f"Correlation (edge_count vs {window_name} Sharpe): r={corr:.4f}, p={pval:.2e}")
    sig = "SIGNIFICANT" if pval < 0.05 else "NOT significant"
    print(f"  -> {sig} at 5% level")
    print()

### 5b. Lagged Cross-Correlation

In [ ]:
def lagged_cross_correlation(x, y, max_lag=40):
    """Compute correlation between x and y at different lags.
    
    Positive lag = x leads y (connectivity predicts future Sharpe)
    Negative lag = y leads x (Sharpe predicts future connectivity)
    """
    x_norm = (x - x.mean()) / x.std()
    y_norm = (y - y.mean()) / y.std()
    
    lags = range(-max_lag, max_lag + 1)
    correlations = []
    p_values = []
    
    for lag in lags:
        if lag > 0:
            corr, pval = stats.pearsonr(x_norm[:-lag], y_norm[lag:])
        elif lag < 0:
            corr, pval = stats.pearsonr(x_norm[-lag:], y_norm[:lag])
        else:
            corr, pval = stats.pearsonr(x_norm, y_norm)
        correlations.append(corr)
        p_values.append(pval)
    
    return np.array(list(lags)), np.array(correlations), np.array(p_values)

# Compute for 60-day rolling Sharpe
lags, xcorr, pvals = lagged_cross_correlation(
    aligned["edge_count"].values, 
    aligned["rolling_sharpe_60"].values,
    max_lag=40,
)

# Plot cross-correlation function
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(lags, xcorr, width=1, color="steelblue", alpha=0.7)
axes[0].axhline(y=0, color="black", lw=0.5)
# Significance bands (approximate: 2/sqrt(N))
n = len(aligned)
sig_band = 2 / np.sqrt(n)
axes[0].axhline(y=sig_band, color="red", ls="--", lw=0.8, label=f"95% CI (±{sig_band:.3f})")
axes[0].axhline(y=-sig_band, color="red", ls="--", lw=0.8)
axes[0].set_xlabel("Lag (days)")
axes[0].set_ylabel("Cross-Correlation")
axes[0].set_title("Lagged Cross-Correlation: Edge Count vs Rolling Sharpe (60-day)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mark peak
peak_lag = lags[np.argmax(np.abs(xcorr))]
peak_corr = xcorr[np.argmax(np.abs(xcorr))]
axes[0].annotate(f"Peak: lag={peak_lag}, r={peak_corr:.3f}",
    xy=(peak_lag, peak_corr), fontsize=10, fontweight="bold",
    arrowprops=dict(arrowstyle="->"), xytext=(peak_lag + 10, peak_corr + 0.05))

# P-values
axes[1].bar(lags, -np.log10(pvals), width=1, color="steelblue", alpha=0.7)
axes[1].axhline(y=-np.log10(0.05), color="red", ls="--", lw=0.8, label="p=0.05")
axes[1].axhline(y=-np.log10(0.01), color="orange", ls="--", lw=0.8, label="p=0.01")
axes[1].set_xlabel("Lag (days)")
axes[1].set_ylabel("-log10(p-value)")
axes[1].set_title("Statistical Significance at Each Lag")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"
Peak cross-correlation: lag={peak_lag} days, r={peak_corr:.4f}")
if peak_lag > 0:
    print(f"  -> Connectivity LEADS Sharpe by {peak_lag} days (predictive)")
elif peak_lag < 0:
    print(f"  -> Sharpe LEADS connectivity by {-peak_lag} days (reverse causation)")
else:
    print(f"  -> Concurrent relationship (lag=0)")

### 5c. Derivative Correlation

In [ ]:
# Compute derivatives (day-over-day changes)
d_edges = aligned["edge_count"].diff().dropna()
d_sharpe_20 = aligned["rolling_sharpe_20"].diff().dropna()
d_sharpe_60 = aligned["rolling_sharpe_60"].diff().dropna()

# Align
d_aligned = pd.DataFrame({
    "d_edges": d_edges,
    "d_sharpe_20": d_sharpe_20,
    "d_sharpe_60": d_sharpe_60,
}).dropna()

for window_name, col in [("20-day", "d_sharpe_20"), ("60-day", "d_sharpe_60")]:
    corr, pval = stats.pearsonr(d_aligned["d_edges"], d_aligned[col])
    print(f"Derivative correlation (d_edges vs d_{window_name}_sharpe): r={corr:.4f}, p={pval:.2e}")
    sig = "SIGNIFICANT" if pval < 0.05 else "NOT significant"
    print(f"  -> {sig} at 5% level")
    print(f"  -> Interpretation: {'turning points align' if abs(corr) > 0.1 else 'turning points do NOT align'}")
    print()

### 5d. Granger Causality Test

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# Test: does lagged edge_count improve prediction of rolling Sharpe?
print("Granger Causality: edge_count -> rolling_sharpe_60")
print("=" * 60)
print("H0: edge_count does NOT Granger-cause rolling_sharpe")
print("If p < 0.05, edge_count has predictive power for Sharpe
")

granger_data = aligned[["rolling_sharpe_60", "edge_count"]].dropna()

try:
    gc_results = grangercausalitytests(granger_data, maxlag=10, verbose=True)
except Exception as e:
    print(f"Error: {e}")
    print("Try with fewer lags or check for stationarity")

## 6. Step 4: Regime Analysis

In [ ]:
# Median split: high vs low connectivity
median_edges = aligned["edge_count"].median()
high_conn = aligned[aligned["edge_count"] >= median_edges]
low_conn = aligned[aligned["edge_count"] < median_edges]

print(f"Median edge count: {median_edges:.0f}")
print(f"High connectivity periods: {len(high_conn)} days")
print(f"Low connectivity periods: {len(low_conn)} days")
print()

# Get corresponding daily returns for each regime
# Align by date with the original daily returns
high_dates = high_conn.index
low_dates = low_conn.index

gcn_high_ret = gcn_returns[gcn_returns.index.isin(high_dates)]
gcn_low_ret = gcn_returns[gcn_returns.index.isin(low_dates)]

from empyrical import sharpe_ratio, annual_return, annual_volatility, max_drawdown

print("GCN Rolling: Regime Analysis")
print("=" * 50)
print(f"{'Metric':<25} {'High Connectivity':>20} {'Low Connectivity':>20}")
print("-" * 65)
print(f"{'Sharpe Ratio':<25} {sharpe_ratio(gcn_high_ret):>20.3f} {sharpe_ratio(gcn_low_ret):>20.3f}")
print(f"{'Annual Return':<25} {annual_return(gcn_high_ret):>20.2%} {annual_return(gcn_low_ret):>20.2%}")
print(f"{'Annual Volatility':<25} {annual_volatility(gcn_high_ret):>20.2%} {annual_volatility(gcn_low_ret):>20.2%}")
print(f"{'Max Drawdown':<25} {-max_drawdown(gcn_high_ret):>20.2%} {-max_drawdown(gcn_low_ret):>20.2%}")
print(f"{'Hit Rate':<25} {(gcn_high_ret > 0).mean():>20.2%} {(gcn_low_ret > 0).mean():>20.2%}")
print(f"{'Num Days':<25} {len(gcn_high_ret):>20} {len(gcn_low_ret):>20}")

In [ ]:
# Increasing vs decreasing connectivity
d_edges_full = aligned["edge_count"].diff()
densifying = aligned[d_edges_full > 0]
sparsifying = aligned[d_edges_full < 0]

gcn_dens_ret = gcn_returns[gcn_returns.index.isin(densifying.index)]
gcn_spars_ret = gcn_returns[gcn_returns.index.isin(sparsifying.index)]

print("
GCN Rolling: Densifying vs Sparsifying")
print("=" * 50)
print(f"{'Metric':<25} {'Densifying':>20} {'Sparsifying':>20}")
print("-" * 65)
print(f"{'Sharpe Ratio':<25} {sharpe_ratio(gcn_dens_ret):>20.3f} {sharpe_ratio(gcn_spars_ret):>20.3f}")
print(f"{'Annual Return':<25} {annual_return(gcn_dens_ret):>20.2%} {annual_return(gcn_spars_ret):>20.2%}")
print(f"{'Annual Volatility':<25} {annual_volatility(gcn_dens_ret):>20.2%} {annual_volatility(gcn_spars_ret):>20.2%}")
print(f"{'Num Days':<25} {len(gcn_dens_ret):>20} {len(gcn_spars_ret):>20}")

In [ ]:
# Bar chart: regime comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# High vs Low connectivity
regime_sharpes = [sharpe_ratio(gcn_high_ret), sharpe_ratio(gcn_low_ret)]
axes[0].bar(["High Connectivity", "Low Connectivity"], regime_sharpes, 
            color=["#d62728", "#1f77b4"], alpha=0.8)
axes[0].set_ylabel("Sharpe Ratio")
axes[0].set_title("GCN Rolling: Sharpe by Connectivity Regime")
axes[0].grid(True, alpha=0.3, axis="y")

# Densifying vs Sparsifying
dens_sharpes = [sharpe_ratio(gcn_dens_ret), sharpe_ratio(gcn_spars_ret)]
axes[1].bar(["Densifying", "Sparsifying"], dens_sharpes,
            color=["#ff7f0e", "#2ca02c"], alpha=0.8)
axes[1].set_ylabel("Sharpe Ratio")
axes[1].set_title("GCN Rolling: Sharpe by Connectivity Direction")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout(); plt.show()

## 7. Step 5: VIX Overlay

In [ ]:
# Download VIX data
try:
    import yfinance as yf
    vix = yf.download("^VIX", start="2017-01-01", end="2023-09-01")["Close"]
    vix.index = pd.to_datetime(vix.index)
    print(f"VIX data: {len(vix)} days, range {vix.index[0].date()} to {vix.index[-1].date()}")
    
    # Align VIX with connectivity
    vix_aligned = vix.reindex(aligned.index, method="ffill")
    
    # Plot
    plot_dual_axis(
        aligned.index, aligned["edge_count"], vix_aligned,
        "Graph Connectivity vs VIX",
        events=events,
    )
    
    # Correlation
    valid = pd.DataFrame({"edges": aligned["edge_count"], "vix": vix_aligned}).dropna()
    corr, pval = stats.pearsonr(valid["edges"], valid["vix"])
    print(f"
Correlation (edge_count vs VIX): r={corr:.4f}, p={pval:.2e}")
    sig = "SIGNIFICANT" if pval < 0.05 else "NOT significant"
    print(f"  -> {sig} at 5% level")
    if corr > 0:
        print(f"  -> Positive: graph densifies during high-VIX (stress) periods")
    else:
        print(f"  -> Negative: graph sparsifies during high-VIX periods")
        
except Exception as e:
    print(f"Could not load VIX data: {e}")
    print("Install yfinance: pip install yfinance")

## 8. Step 6: GCN vs GAT Comparison

In [ ]:
# Repeat key analyses for GAT 4c
if gat_adj is not None:
    gat_edges = compute_edge_counts(gat_adj)
    gat_sharpe_60 = compute_rolling_sharpe(gat_returns, 60)
    
    gat_connectivity = pd.Series(gat_edges, index=pd.to_datetime(gat_dates), name="edge_count")
    gat_aligned = pd.DataFrame({
        "edge_count": gat_connectivity,
        "rolling_sharpe_60": gat_sharpe_60,
    }).dropna()
    
    # GCN vs GAT connectivity overlay
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(aligned.index, aligned["edge_count"], label="GCN (Pearson edges)", lw=0.8)
    if len(gat_aligned) > 0:
        ax.plot(gat_aligned.index, gat_aligned["edge_count"], label="GAT (same Pearson mask)", lw=0.8, alpha=0.7)
    ax.set_ylabel("Number of Edges")
    ax.set_title("GCN vs GAT: Graph Connectivity Over Time")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    print("Note: GCN and GAT use the same Pearson mask, so edge counts should be identical.")
    print("The difference is in edge WEIGHTS (attention vs Pearson correlation values).")

# If attention weights available, compute mean attention weight over time
if gat_attn is not None:
    # Average attention across heads
    gat_attn_avg = gat_attn.mean(axis=1)  # (windows, 88, 88)
    
    # Mean attention weight within the Pearson mask over time
    if gat_adj is not None:
        mean_attn_in_mask = []
        for t in range(len(gat_attn_avg)):
            mask = gat_adj[t] > 0
            if mask.sum() > 0:
                mean_attn_in_mask.append(gat_attn_avg[t][mask].mean())
            else:
                mean_attn_in_mask.append(np.nan)
        
        mean_attn_series = pd.Series(mean_attn_in_mask, index=pd.to_datetime(gat_dates))
        
        fig, ax1 = plt.subplots(figsize=(16, 5))
        ax1.plot(mean_attn_series.index, mean_attn_series.values, lw=0.8, color="#1f77b4")
        ax1.set_ylabel("Mean Attention Weight (within Pearson mask)")
        ax1.set_xlabel("Date")
        ax1.set_title("4c GAT: Mean Learned Attention Weight Over Time")
        ax1.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
        
        print(f"Mean attention in mask: {np.nanmean(mean_attn_in_mask):.4f}")
        print(f"Std: {np.nanstd(mean_attn_in_mask):.4f}")
else:
    print("GAT attention weights not available. Run 4c with save_experiment_results() first.")

## 9. Summary

In [ ]:
print("=" * 60)
print("5b CONNECTIVITY ANALYSIS SUMMARY")
print("=" * 60)

# Level correlation
corr_level, pval_level = stats.pearsonr(aligned["edge_count"], aligned["rolling_sharpe_60"])
print(f"
Level correlation (edges vs 60d Sharpe): r={corr_level:.4f}, p={pval_level:.2e}")

# Derivative correlation
d_aligned_clean = d_aligned.dropna()
corr_deriv, pval_deriv = stats.pearsonr(d_aligned_clean["d_edges"], d_aligned_clean["d_sharpe_60"])
print(f"Derivative correlation: r={corr_deriv:.4f}, p={pval_deriv:.2e}")

# Peak lag
print(f"Peak cross-correlation lag: {peak_lag} days (r={peak_corr:.4f})")

# Regime
print(f"
Regime analysis:")
print(f"  High connectivity Sharpe: {sharpe_ratio(gcn_high_ret):.3f}")
print(f"  Low connectivity Sharpe:  {sharpe_ratio(gcn_low_ret):.3f}")

print("
Key findings:")
if abs(corr_level) > 0.1 and pval_level < 0.05:
    direction = "positive" if corr_level > 0 else "negative"
    print(f"  - Statistically significant {direction} relationship between connectivity and performance")
else:
    print(f"  - No significant level relationship (visual alignment may be coincidental)")

if abs(corr_deriv) > 0.1 and pval_deriv < 0.05:
    print(f"  - Turning points in connectivity align with turning points in Sharpe")
else:
    print(f"  - Turning points do NOT significantly align")